In [1]:
!pip install -q -U transformers sentencepiece sacremoses datasets accelerate evaluate bert-score nltk pandas plotly


In [ ]:
import os
import time
import json
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from nltk.translate.bleu_score import corpus_bleu
from bert_score import score as bertscore_fn
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

nltk.download('punkt')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SRC_LANG = 'san_Deva'
TGT_LANG = 'eng_Latn'
MODEL_NAME = 'facebook/nllb-200-distilled-600M'
DATA_DIR = Path('./dataset')
OUTPUT_DIR = Path('./artifacts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LEN = 128
EMBED_TRAIN_FILE = OUTPUT_DIR / 'train_embeddings.npz'
EMBED_DEV_FILE = OUTPUT_DIR / 'dev_embeddings.npz'
EMBED_TEST_FILE = OUTPUT_DIR / 'test_embeddings.npz'
TRAIN_LOG_CSV = OUTPUT_DIR / 'training_log.csv'
SUBMISSION_FILE = OUTPUT_DIR / 'submission.csv'
METRICS_FILE = OUTPUT_DIR / 'metrics.json'

print({'device': DEVICE, 'model': MODEL_NAME})


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


{'device': 'cuda', 'model': 'facebook/nllb-200-distilled-600M'}


In [3]:
def load_split(split: str, count: str) -> pd.DataFrame:
    sa = pd.read_csv(DATA_DIR / f'{split}_sa_{count}.csv')
    path_en = DATA_DIR / f'{split}_en_{count}.csv'
    if path_en.exists():
        en = pd.read_csv(path_en)
        df = sa.merge(en, on='Source_id', how='inner')
    else:
        df = sa.copy()
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna('').astype(str).str.strip()
    return df

train_df = load_split('train', '10000')
dev_df = load_split('dev', '1000')
test_df = load_split('test', '1000')
print('train/dev/test:', train_df.shape, dev_df.shape, test_df.shape)


train/dev/test: (10000, 3) (1000, 3) (1000, 3)


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=SRC_LANG)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

forced_bos_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
model.generation_config.forced_bos_token_id = forced_bos_id
model.generation_config.max_new_tokens = MAX_LEN
model.generation_config.num_beams = 5
model.generation_config.early_stopping = True

total_params = sum(p.numel() for p in model.parameters())
hidden_size = getattr(model.config, 'd_model', None) or getattr(model.config, 'hidden_size', None)
print(f'Total parameters: {total_params:,}')
print(f'Embedding dim: {hidden_size}')


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Total parameters: 615,073,792
Embedding dim: 1024


In [5]:
@torch.no_grad()
def mean_pool_last_hidden_state(texts, batch_size=16, max_length=MAX_LEN):
    model.eval()
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = [str(x).strip() for x in texts[i:i+batch_size]]
        enc = tokenizer(
            batch,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(DEVICE)
        outputs = model.model.encoder(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask'],
            return_dict=True,
        )
        hidden = outputs.last_hidden_state
        mask = enc['attention_mask'].unsqueeze(-1)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        all_embeddings.append(pooled.detach().cpu())
    return torch.cat(all_embeddings, dim=0).numpy()


def save_embeddings(npz_path: Path, df: pd.DataFrame, text_col: str):
    vectors = mean_pool_last_hidden_state(df[text_col].tolist())
    np.savez_compressed(
        npz_path,
        source_id=df['Source_id'].to_numpy(),
        text=df[text_col].to_numpy(),
        embeddings=vectors,
    )
    return vectors.shape


def ensure_embeddings(npz_path: Path, df: pd.DataFrame, text_col: str, split_name: str):
    if npz_path.exists():
        cache = np.load(npz_path, allow_pickle=True)
        print(f'Using cached {split_name} embeddings:', cache['embeddings'].shape)
        return cache
    shape = save_embeddings(npz_path, df, text_col)
    print(f'Created {split_name} embeddings:', shape)
    return np.load(npz_path, allow_pickle=True)


In [6]:
train_emb = ensure_embeddings(EMBED_TRAIN_FILE, train_df, 'Sentence_sa', 'train')
dev_emb = ensure_embeddings(EMBED_DEV_FILE, dev_df, 'Sentence_sa', 'dev')
test_emb = ensure_embeddings(EMBED_TEST_FILE, test_df, 'Sentence_sa', 'test')


Using cached train embeddings: (10000, 1024)
Using cached dev embeddings: (1000, 1024)
Using cached test embeddings: (1000, 1024)


In [7]:
tokenizer.src_lang = SRC_LANG

@torch.no_grad()
def translate(sentences, batch_size=16, num_beams=5, max_new_tokens=128):
    model.eval()
    prev_beams = model.generation_config.num_beams
    prev_tokens = model.generation_config.max_new_tokens
    out = []
    model.generation_config.num_beams = num_beams
    model.generation_config.max_new_tokens = max_new_tokens
    try:
        for i in range(0, len(sentences), batch_size):
            batch = [str(x).strip() for x in sentences[i:i+batch_size]]
            enc = tokenizer(
                batch,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=MAX_LEN,
            ).to(DEVICE)
            gen = model.generate(**enc)
            out.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
    finally:
        model.generation_config.num_beams = prev_beams
        model.generation_config.max_new_tokens = prev_tokens
    return out


In [8]:
if DEVICE == 'cuda':
    torch.cuda.synchronize()
dev_start = time.time()
dev_preds = translate(dev_df['Sentence_sa'].tolist())
if DEVICE == 'cuda':
    torch.cuda.synchronize()
dev_inference_time = time.time() - dev_start

if DEVICE == 'cuda':
    torch.cuda.synchronize()
test_start = time.time()
test_preds = translate(test_df['Sentence_sa'].tolist())
if DEVICE == 'cuda':
    torch.cuda.synchronize()
test_inference_time = time.time() - test_start

submission = pd.DataFrame({
    'Source_id': test_df['Source_id'],
    'Sentence_en': test_preds,
})
submission.to_csv(SUBMISSION_FILE, index=False, encoding='utf-8')

references = [[ref.split()] for ref in dev_df['Sentence_en'].tolist()]
hypotheses = [pred.split() for pred in dev_preds]
bleu_score = corpus_bleu(references, hypotheses)
P, R, F1 = bertscore_fn(dev_preds, dev_df['Sentence_en'].tolist(), lang='en', rescale_with_baseline=True)

metrics = {
    'BLEU': float(bleu_score),
    'BERTScore F1': float(F1.mean().item()),
    'Dev time (s)': float(dev_inference_time),
    'Test time (s)': float(test_inference_time),
    'Parameter count': int(total_params),
}
with open(METRICS_FILE, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

pd.DataFrame([metrics])


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/bert_score/score.py:149: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before

,BLEU,BERTScore F1,Dev time (s),Test time (s),Parameter count
0,0.126886,0.488385,168.453276,165.897295,615073792


In [9]:
print('Generated files:')
for p in sorted(OUTPUT_DIR.rglob('*')):
    print(p)


Generated files:
artifacts/dev_embeddings.npz
artifacts/metrics.json
artifacts/submission.csv
artifacts/test_embeddings.npz
artifacts/train_embeddings.npz
